# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is a Croissant JSON-LD schema accessible at the following URL:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset Title: {getattr(dataset.metadata, 'name', 'No title')}")
print(f"\nDescription: {getattr(dataset.metadata, 'description', 'No description')}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and columns.

In [ ]:
# We'll list out all available record sets and their fields by @id

record_sets = dataset.record_sets
if not record_sets:
    print("No record sets defined in the Croissant schema.")
else:
    for rs in record_sets:
        print(f"RecordSet: @id={rs.id}  name={getattr(rs, 'name', '(no name)')}")
        if hasattr(rs, 'fields') and rs.fields:
            for f in rs.fields:
                print(f"    Field: @id={f.id}, name={getattr(f, 'name', '(no name)')}, type={getattr(f, 'data_type', '(no type)')}")
        if hasattr(rs, 'columns') and rs.columns:
            for col in rs.columns:
                print(f"    Column: @id={col.id}, name={getattr(col, 'name', '(no name)')}, type={getattr(col, 'data_type', '(no type)')}")

# Let's also get the first record set's id for following sections (if present)
first_record_set_id = None
if record_sets:
    first_record_set_id = record_sets[0].id
    print(f"\nFirst RecordSet @id for later use: {first_record_set_id}")

## 3. Data Extraction
Load data from available record set(s) into pandas DataFrames for analysis. Each record set can be loaded by referencing its `@id`.

In [ ]:
# Extract data from each available record set into a DataFrame

dfs = {}
record_set_ids = [rs.id for rs in record_sets] if record_sets else []
for rs_id in record_set_ids:
    try:
        recs = list(dataset.records(record_set=rs_id))
        if recs:
            dfs[rs_id] = pd.DataFrame(recs)
            print(f"Loaded {len(recs)} records from RecordSet @id={rs_id}.")
            print("Columns:", dfs[rs_id].columns.tolist())
        else:
            print(f"RecordSet {rs_id} produced no records.")
    except Exception as e:
        print(f"Error loading RecordSet {rs_id}: {e}")

# Display the head of the first loaded DataFrame (if any)
if dfs:
    sample_rs_id = list(dfs.keys())[0]
    display(dfs[sample_rs_id].head())
else:
    print("No tabular record sets available to load as DataFrame.")

## 4. Exploratory Data Analysis (EDA)
Let's apply basic analysis by selecting a numeric field (by its `@id`) from a loaded record set, filtering, normalizing, and performing a groupby if a categorical/group field exists.

In [ ]:
# For demonstration, we'll try on the first DataFrame available
if dfs:
    df = dfs[sample_rs_id]
    print(f"Available columns in RecordSet {sample_rs_id}:")
    print(df.columns.tolist())
    
    # Try to infer a numeric field by dtype
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using field '@id' as numeric: {numeric_field_id}")
        threshold = df[numeric_field_id].mean()  # Example threshold: mean value
        filtered = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered.head())
        # Normalization
        filtered[f"{numeric_field_id}_normalized"] = (filtered[numeric_field_id] - filtered[numeric_field_id].mean()) / filtered[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try to pick a group field (categorical/text field that's not id or a number)
        candidate_groups = [col for col in df.columns if (df[col].dtype == object and col != numeric_field_id)]
        if candidate_groups:
            group_field_id = candidate_groups[0]
            print(f"Grouping by '{group_field_id}':")
            grouped = filtered.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize the numeric field distribution and, if possible, an example group comparison.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Visualizations
if dfs and numeric_cols:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color='skyblue')
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if 'group_field_id' in locals():
        plt.figure(figsize=(9, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
This notebook has demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library. We examined available record sets and fields (by their `@id`), loaded data into pandas, performed initial EDA, and visualized key numeric variables. 

**Key findings:**
- Dataset fields and structure are discoverable by `@id`.
- Numeric and categorical fields can be identified and analyzed using standard data science workflows.
- Further analysis can now be performed according to your research or application needs.

For additional information, consult the [FAIR^2 Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) and the [mlcroissant documentation](https://mlcommons.github.io/croissant/).
